In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv")

In [ ]:
data['document_type'].value_counts()

In [ ]:
data = data[data['document_type']=='vermögensverzeichnis']

In [ ]:
data

In [ ]:
# add intent recog as module
import sys
sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")
from intent_recognition.src.utils import get_jsons_from_s3


In [ ]:
# get my local data with invoice and without invoice
data = data[data['object_key'].str.contains("data/aftercourt", na=False)]
data

In [ ]:
ve_with_invoice = data[data['is_ve_with_invoice'] == True]
ve_without_invoice = data[data['is_ve_with_invoice'] == False]

In [ ]:
ve_with_invoice

In [ ]:
ve_without_invoice

In [ ]:
s3_bucket_name = "pair-email-classification"
key_without_invoice = "data/aftercourt/vermögensverzeichnis_without_invoice/"
key_with_invoice = "data/aftercourt/vermögensverzeichnis_with_invoice/"

In [ ]:
from utils.use_textract_utils import _get_textract_client, wait_for_job_completion, get_texts_from_textract_outputs

In [ ]:

object_key = ve_with_invoice['object_key'].iloc[0]
textract = _get_textract_client()
job_id = textract.submit_textract_job(s3_bucket_name, object_key)

job_info = {
            'job_id': job_id,
            'doc_key': object_key,
            'status': 'SUBMITTED'
        }

In [ ]:
textract_output = wait_for_job_completion(textract, job_info)

In [ ]:
textract_output

In [ ]:

raw_text = []
for cur_doc in textract_output['result']:
    if cur_doc['BlockType'] == "LINE":
        raw_text.append(cur_doc["Text"])    
    
all_text = "\n".join(raw_text)

print(all_text)

In [ ]:
textract_output.keys()

In [ ]:
result = textract_output['result']
type(result)

In [ ]:
unique_block_types = set([cur_doc['BlockType'] for cur_doc in result])
print(unique_block_types)

In [ ]:
for block in result:
    if block['BlockType'] == "PAGE":
        page_block = block
        break
page_block.keys()

In [ ]:
for block in result:
    if block['BlockType'] == "WORD":
        word_block = block
        break
word_block.keys()

In [ ]:
for block in result:
    if block['BlockType'] == "LINE":
        line_block = block
        break
line_block.keys()

## check content of page block type

In [ ]:
page_block
# relationships contains ids of line block type

In [ ]:
line_block # relationships contains ids of word block type

In [ ]:
word_block
#smallest element

In [ ]:
distinct_words = set()
for block in result:
    if block['BlockType'] == "WORD":
        distinct_words.add(block['Text'])

In [ ]:
pages_to_line_dict = {}
for block in result:
    if block['BlockType'] == "LINE":
        page_number = block['Page']
        page_id = block['Id']
        line_ids = []
        for relationship in block.get('Relationships', []):
            if relationship['Type'] == "CHILD":
                line_ids.extend(relationship['Ids'])
        pages_to_line_dict[page_id] = line_ids

In [ ]:
page_id_to_page_number_dict = {}
for block in result:
    if block['BlockType'] == "PAGE":
        page_id = block['Id']
        page_number = block['Page']
        page_id_to_page_number_dict[page_id] = page_number
page_id_to_page_number_dict

In [ ]:
# sort page_id_to_page_number_dict
page_id_to_page_number_dict = dict(sorted(page_id_to_page_number_dict.items(), key=lambda item: item[1]))
page_id_to_page_number_dict

In [ ]:
# import defaultdict
from collections import defaultdict

pages_to_text_dict = defaultdict(list[str])
for block in result:
    if block['BlockType'] == "LINE":
        page_number = block['Page']
        text = block["Text"]
        pages_to_text_dict[page_number].append(text)

In [ ]:
pages_to_text_dict.keys()

In [ ]:
# merge limes with \n
for page_number, lines in pages_to_text_dict.items():
    pages_to_text_dict[page_number] = "\n".join(lines)

In [ ]:
# first page
print(pages_to_text_dict[2])

In [ ]:
import pandas as pd
labels_path= "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/classification/vermögensverzeichnis/invoice_page_labels.txt"

invoice_page_labels = pd.read_csv(labels_path, sep=",", header=None, names=["ticket_uuid", "invoice_page_start", "invoice_page_end"])
# drop first row
invoice_page_labels = invoice_page_labels.drop(0)
invoice_page_labels

# create page-by-page dataset

In [ ]:
# add intent recog as module
import sys
sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")
from intent_recognition.src.utils import get_jsons_from_s3


In [ ]:
data = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv")
data = data[data['document_type']=='vermögensverzeichnis']
# get my local data with invoice and without invoice
data = data[data['object_key'].str.contains("data/aftercourt", na=False)]
ve_with_invoice = data[data['is_ve_with_invoice'] == True]
ve_without_invoice = data[data['is_ve_with_invoice'] == False]

In [ ]:
data

In [ ]:
s3_bucket_name = "pair-email-classification"
key_without_invoice = "data/aftercourt/vermögensverzeichnis_without_invoice/"
key_with_invoice = "data/aftercourt/vermögensverzeichnis_with_invoice/"

In [ ]:
from utils.use_textract_utils import _get_textract_client, wait_for_job_completion, get_texts_from_textract_outputs

In [ ]:
textract_outputs = []
textract = _get_textract_client()
for object_key in data['object_key']:
    job_id = textract.submit_textract_job(s3_bucket_name, object_key)

    job_info = {
                'job_id': job_id,
                'doc_key': object_key,
                'status': 'SUBMITTED'
            }
    textract_output = wait_for_job_completion(textract, job_info)
    textract_outputs.append(textract_output)

In [ ]:
page_by_page_texts = []

for idx, row in data.reset_index(drop=True).iterrows():
    ticket_uuid = row['ticket_uuid']
    object_key = row['object_key']
    document_type = row['document_type']
    is_ve_with_invoice = row['is_ve_with_invoice']
    try:
        textract_output = textract_outputs[idx]
    except IndexError:
        print(f"Index: {idx}, IndexError for ticket_uuid: {ticket_uuid}, object_key: {object_key}")
        continue
    status = textract_output['status']
    if status != "SUCCEEDED":
        print(f"Textract job did not succeed for ticket_uuid: {ticket_uuid}, object_key: {object_key}, status: {status}")
        continue
    
    current_result = textract_output['result']
    from collections import defaultdict

    pages_to_text_dict = defaultdict(list[str])
    for block in current_result:
        if block['BlockType'] == "LINE":
            page_number = block['Page']
            text = block["Text"]
            pages_to_text_dict[page_number].append(text)
    
    for page_number, lines in pages_to_text_dict.items():
        pages_to_text_dict[page_number] = "\n".join(lines)
    page_by_page_texts.append({
        "ticket_uuid": ticket_uuid,
        "document_type": document_type,
        "is_ve_with_invoice": is_ve_with_invoice,
        "object_key": object_key,
        "pages_to_text_dict": pages_to_text_dict
    })


In [ ]:
# create df
page_by_page_texts = pd.DataFrame(page_by_page_texts)

In [ ]:
#page_by_page_texts.to_csv("tmp.csv", index=False)

In [ ]:
page_by_page_texts

In [ ]:
ve_invoice_data_with_labels = page_by_page_texts.merge(invoice_page_labels, on="ticket_uuid", how="left")

In [ ]:
ve_invoice_data_with_labels

In [ ]:
ve_invoice_data_with_labels

In [ ]:
ve_invoice_data_with_labels['is_ve_with_invoice'].value_counts()

In [ ]:
ticket_uuid_to_invoice_start_text = {}
for row in ve_invoice_data_with_labels.itertuples():
    is_ve_with_invoice = row.is_ve_with_invoice
    if is_ve_with_invoice:
        start_page = int(row.invoice_page_start)
        invoice_start_text = row.pages_to_text_dict.get(start_page, "")
        ticket_uuid_to_invoice_start_text[row.ticket_uuid] = invoice_start_text
    else:
        # no invoice inside ve, skip for now
        continue

In [ ]:
ticket_uuid_to_invoice_start_text

In [ ]:
from utils.intent_recog_utils import apply_text_cleaning

# Apply the same text cleaning used across the repo to each page's text.
ticket_uuid_to_invoice_start_text_cleaned = {
    ticket_uuid: apply_text_cleaning(text or "")
    for ticket_uuid, text in ticket_uuid_to_invoice_start_text.items()
}
ticket_uuid_to_invoice_start_text_cleaned

In [ ]:
from collections import Counter

from utils.intent_recog_utils import apply_tokenization

# Tokenize each cleaned invoice start page using the repo's standard tokenizer
# (spaCy lemmas, stopword/punct/number filtering — same pipeline used for classification).
invoice_start_tokens_per_doc = {
    ticket_uuid: apply_tokenization(text)
    for ticket_uuid, text in ticket_uuid_to_invoice_start_text_cleaned.items()
    if text
}

# Total token frequency across all invoice start pages.
token_counter = Counter()
for tokens in invoice_start_tokens_per_doc.values():
    token_counter.update(tokens)

# Document frequency: in how many invoice start pages does each token appear?
doc_freq_counter = Counter()
for tokens in invoice_start_tokens_per_doc.values():
    doc_freq_counter.update(set(tokens))

n_docs = len(invoice_start_tokens_per_doc)
print(f"Analyzed {n_docs} invoice start pages | unique tokens: {len(token_counter)}")

In [ ]:
top_n = 40
top_tokens_df = pd.DataFrame(
    [
        {
            "token": tok,
            "total_count": cnt,
            "doc_count": doc_freq_counter[tok],
            "doc_ratio": doc_freq_counter[tok] / n_docs if n_docs else 0.0,
        }
        for tok, cnt in token_counter.most_common(top_n)
    ]
)
top_tokens_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(top_tokens_df))))
ax.barh(top_tokens_df["token"][::-1], top_tokens_df["total_count"][::-1], color="#4C78A8")
ax.set_xlabel("Total occurrences")
ax.set_title(f"Top {len(top_tokens_df)} keywords on invoice start pages (n={n_docs})")
plt.tight_layout()
plt.show()

## What distinguishes an invoice start page?

Compare token distributions between **invoice start pages** (positive class) and
**all other pages** (negative class: every non-start page from invoice VEs +
every page from non-invoice VEs). Tokens with high positive lift / log-odds
are good signal candidates for detecting the invoice start page.

In [ ]:
import math

from utils.intent_recog_utils import apply_text_cleaning, apply_tokenization

# Build positive (invoice-start) vs negative (every other page) corpora.
# Positive set is already computed in `invoice_start_tokens_per_doc`.
positive_tokens_per_page = list(invoice_start_tokens_per_doc.values())

negative_tokens_per_page = []
for row in ve_invoice_data_with_labels.itertuples():
    pages_dict = row.pages_to_text_dict
    if row.is_ve_with_invoice:
        try:
            start_page = int(row.invoice_page_start)
            end_page = int(row.invoice_page_end)
        except (TypeError, ValueError):
            start_page = None
            end_page = None
    else:
        start_page = None  # every page is "non-start"
        end_page = None

    for page_num, page_text in pages_dict.items():
        if start_page is not None and int(page_num) == start_page and end_page is not None and int(page_num) == end_page:
            continue  # skip the invoice start page itself
        if not page_text:
            continue
        cleaned = apply_text_cleaning(page_text)
        tokens = apply_tokenization(cleaned)
        if tokens:
            negative_tokens_per_page.append(tokens)

n_pos = len(positive_tokens_per_page)
n_neg = len(negative_tokens_per_page)
print(f"Positive (invoice start) pages: {n_pos}")
print(f"Negative (other) pages:         {n_neg}")

In [ ]:
# Document-frequency counts in each class.
pos_doc_freq = Counter()
for toks in positive_tokens_per_page:
    pos_doc_freq.update(set(toks))

neg_doc_freq = Counter()
for toks in negative_tokens_per_page:
    neg_doc_freq.update(set(toks))

# Discriminative score per token using:
#   - p_pos: P(token appears | invoice start page)
#   - p_neg: P(token appears | other page)
#   - lift:  p_pos / p_neg  (how much more likely on start pages)
#   - log_odds with Laplace smoothing
SMOOTH = 0.5
vocab = set(pos_doc_freq) | set(neg_doc_freq)

rows = []
for tok in vocab:
    pf = pos_doc_freq.get(tok, 0)
    nf = neg_doc_freq.get(tok, 0)
    p_pos = pf / n_pos if n_pos else 0.0
    p_neg = nf / n_neg if n_neg else 0.0
    log_odds = math.log(((pf + SMOOTH) / (n_pos - pf + SMOOTH)) /
                        ((nf + SMOOTH) / (n_neg - nf + SMOOTH)))
    rows.append({
        "token": tok,
        "pos_doc_count": pf,
        "neg_doc_count": nf,
        "p_pos": p_pos,
        "p_neg": p_neg,
        "lift": (p_pos / p_neg) if p_neg > 0 else float("inf"),
        "log_odds": log_odds,
    })

discriminative_df = pd.DataFrame(rows)

# Filter out very rare tokens to avoid noise.
MIN_POS_DOC_COUNT = max(3, int(0.1 * n_pos))
filtered = discriminative_df[discriminative_df["pos_doc_count"] >= MIN_POS_DOC_COUNT].copy()

print(f"Vocab size: {len(discriminative_df)} | after min_pos_doc_count >= {MIN_POS_DOC_COUNT}: {len(filtered)}")

In [ ]:
# Top tokens that *appear much more often* on invoice start pages.
top_distinctive = (
    filtered.sort_values("log_odds", ascending=False)
    .head(40)
    .reset_index(drop=True)
)
top_distinctive

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(top_distinctive))))
labels = [
    f"{t}  (pos {p}/{n_pos}  vs  neg {n}/{n_neg})"
    for t, p, n in zip(
        top_distinctive["token"],
        top_distinctive["pos_doc_count"],
        top_distinctive["neg_doc_count"],
    )
]
ax.barh(labels[::-1], top_distinctive["log_odds"][::-1], color="#2CA02C")
ax.set_xlabel("Log-odds (positive = invoice-start indicator)")
ax.set_title("Most distinctive tokens for invoice start pages")
plt.tight_layout()
plt.show()

# Create dataset from production

idea: we are running is invoice inside in the pfub model
using this, fetch some data that is invoice inside

then run basic vermogenverzeichnis check to extract with invoice. based on that create a dataset

In [ ]:
import os
import json
from ast import literal_eval
from datetime import datetime, timedelta

import pandas as pd

import numpy as np
from IPython.display import clear_output

from python_utilities.db_connection import DbConnection

analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')

In [ ]:
query_invoice = """
SELECT *
FROM (llm_attachments_predictions lpp 
LEFT JOIN llm_attachments la
ON lpp.attachment_id = la.attachment_id) 
LEFT JOIN textract_jobs tj
ON lpp.attachment_id = tj.attachment_id
WHERE lpp.subtype = 'is_invoice_inside'
AND lpp.value LIKE '%%True%%'
LIMIT 5000
"""
invoice_data = analytics_db.sql_to_df(query_invoice)

In [ ]:
not_invoice_data = analytics_db.sql_to_df("""
SELECT *
FROM (llm_attachments_predictions lpp 
LEFT JOIN llm_attachments la
ON lpp.attachment_id = la.attachment_id) 
LEFT JOIN textract_jobs tj
ON lpp.attachment_id = tj.attachment_id
WHERE lpp.subtype = 'is_invoice_inside'
AND lpp.value NOT LIKE '%%True%%'
LIMIT 5000
""")

In [ ]:
invoice_data

In [ ]:
from typing import Any
def get_jsons_from_s3_parallel_custom(
    s3_links: list[str],
    max_workers: int = 20,
) -> list[dict[str, Any]]:
    """Download Textract JSONs from S3 in parallel using a thread pool.

    Returns results in the same order as ``s3_links``.
    Failed downloads return a fallback empty-line object.
    """
    import boto3
    from botocore.exceptions import ClientError
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import json
    import logging

    logger = logging.getLogger(__name__)
    session = boto3.Session(profile_name="739275445236_DataScienceUser")
    s3_client = session.client("s3")
    fallback = [{"BlockType": "LINE", "Text": ""}]

    def _download_one(link: str) -> dict[str, Any]:
        try:
            parts = link.replace("s3://", "").split("/")
            bucket_name = parts[0]
            key = "/".join(parts[1:])
            response = s3_client.get_object(Bucket=bucket_name, Key=key)
            content = response["Body"].read().decode("utf-8")
            return json.loads(content)
        except ClientError as e:
            logger.error(f"Error retrieving object from {link}: {e}")
            return fallback
        except json.JSONDecodeError:
            logger.error(f"Error decoding JSON from {link}")
            return fallback
        except Exception as e:
            logger.error(f"Error from {link} {e}")
            return fallback

    results: list[dict[str, Any] | None] = [None] * len(s3_links)
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {
            pool.submit(_download_one, link): idx
            for idx, link in enumerate(s3_links)
        }
        for future in as_completed(futures):
            results[futures[future]] = future.result()
    return results

In [ ]:

from collections import defaultdict

# Keep only rows that have a Textract output stored in S3
invoice_data_with_s3 = invoice_data.dropna(subset=["s3_link"]).reset_index(drop=True)

s3_links = invoice_data_with_s3["s3_link"].tolist()
textract_jsons = get_jsons_from_s3_parallel_custom(s3_links)

def blocks_to_pages_text(blocks: list) -> dict:
    pages: dict = defaultdict(list)
    for block in blocks:
        if block.get("BlockType") == "LINE":
            pages[block["Page"]].append(block["Text"])
    return {page: "\n".join(lines) for page, lines in pages.items()}

invoice_data_with_s3["textract_blocks"] = textract_jsons


In [ ]:
invoice_data_with_s3

In [ ]:
invoice_data_with_s3


In [ ]:
invoice_data_with_s3['number_of_pages'] = invoice_data_with_s3['textract_blocks'].apply(lambda blocks: len(set(block['Page'] for block in blocks)))

In [ ]:
invoice_data_with_s3['number_of_pages'].value_counts()

In [ ]:
invoice_data_with_s3.to_csv("invoice_data_with_s3.csv", index=False)